In [18]:
import os
import xml.etree.ElementTree as ET

def check_lattice_free_energy(directory="."):
    """Find folders with complete vasprun.xml and print incomplete ones."""
    # Check if the user asked for help
    if directory == "help":
        print("Please use this function on the parent directory of the project's main folder.")
        return []

    complete_folders = []
    # Traverse all folders under "directory"
    for dirpath, _, filenames in os.walk(directory):
        if "vasprun.xml" in filenames:
            file_name_xml = os.path.join(dirpath, "vasprun.xml")

            # Check if vasprun.xml is complete
            try:
                with open(file_name_xml, "r", encoding="utf-8") as f:
                    # Check the last few lines for the closing tag
                    last_lines = f.readlines()[-10:]    # read the last 10 lines
                    for line in last_lines:
                        if "</modeling>" in line or "</vasp>" in line:
                            complete_folders.append(dirpath)
                            break
                    else:
                        print(f"vasprun.xml in {dirpath} is incomplete.")
            except IOError as e:                        # Change from Exception to IOError
                print(f"Error reading {file_name_xml}: {e}")

    return complete_folders

def summarize_free_energy_directory(directory=".", lattice_start=None, lattice_end=None):
    result_file = "lattice_free_energy.dat"
    result_file_path = os.path.join(directory, result_file)
    
    if directory == "help":
        print("Please use this function on the parent directory of the project's main folder.")
        return []
    
    # Use check_lattice_free_energy to get folders with complete vasprun.xml
    dirs_to_walk = check_lattice_free_energy(directory)
    results = []

    for dest_dir in dirs_to_walk:
        file_name_xml = os.path.join(dest_dir, "vasprun.xml")

        if os.path.isfile(file_name_xml):
            # Extract e_fr_energy from vasprun.xml
            tree = ET.parse(file_name_xml)
            root = tree.getroot()
            e_fr_energy = float(root.findall(".//calculation/energy/i[@name='e_fr_energy']")[-1].text)
            basis_vectors = root.findall(".//calculation/structure/crystal/varray[@name='basis']")[-1]
            a_vector = basis_vectors[0].text.split()
            a_length = (float(a_vector[0])**2 + float(a_vector[1])**2 + float(a_vector[2])**2)**0.5
            lattice_constant = a_length

            # Check if lattice constant is within the specified range
            TOLERANCE = 1e-6
            within_start = lattice_start is None or lattice_constant >= lattice_start - TOLERANCE
            within_end = lattice_end is None or lattice_constant <= lattice_end + TOLERANCE
            if within_start and within_end:
                results.append((lattice_constant, e_fr_energy))

    # Sort the results by lattice_constant (the first element of the tuple)
    results.sort(key=lambda x: x[0])

    # Now write the sorted results to the file
    with open(result_file_path, "w", encoding="utf-8") as f:
        f.write("Lattice\t Free energy\n")
        for lattice_constant, e_fr_energy in results:
            f.write(f"{lattice_constant:.3f}\t{e_fr_energy}\n")

summarize_free_energy_directory(".", 4.840, 4.860)